In [1]:
import numpy as np

In [2]:
class Matrix:
    def __init__(self, data):
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows, self.cols)

    def __repr__(self):
        col_widths = []
        for j in range(self.cols):
            width = max(len(f"{self.data[i][j]:.4f}") for i in range(self.rows))
            col_widths.append(width)
        lines = []
        for i in range(self.rows):
            row_str = "  ".join(
                f"{self.data[i][j]:{col_widths[j]}.4f}" for j in range(self.cols)
            )
            bracket_l = "|" if 0 < i < self.rows - 1 else ("/" if i == 0 else "\\")
            bracket_r = "|" if 0 < i < self.rows - 1 else ("\\" if i == 0 else "/")
            lines.append(f"  {bracket_l} {row_str} {bracket_r}")
        header = f"Matrix {self.rows}x{self.cols}:"
        return header + "\n" + "\n".join(lines)

    def scalar_multiply(self, scalar):
        return Matrix([
            [self.data[i][j] * scalar for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def transpose(self):
        return Matrix([
            [self.data[j][i] for j in range(self.rows)]
            for i in range(self.cols)
        ])

    @property
    def T(self):
        return self.transpose()

    def determinant(self):
        if self.rows != self.cols:
            raise ValueError("Determinant only defined for square matrices")
        if self.shape == (1, 1):
            return self.data[0][0]
        if self.shape == (2, 2):
            return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
        det = 0
        for j in range(self.cols):
            minor = Matrix([
                [self.data[i][k] for k in range(self.cols) if k != j]
                for i in range(1, self.rows)
            ])
            det += ((-1) ** j) * self.data[0][j] * minor.determinant()
        return det

    def inverse_2x2(self):
        if self.shape != (2, 2):
            raise ValueError("This method only works for 2x2 matrices")
        det = self.determinant()
        if abs(det) < 1e-10:
            raise ValueError("Matrix is singular, no inverse exists")
        return Matrix([
            [self.data[1][1] / det, -self.data[0][1] / det],
            [-self.data[1][0] / det, self.data[0][0] / det]
        ])

    def inverse(self):
        minors = Matrix([
            [
                Matrix([
                [self.data[n][k] for k in range(self.cols) if k != j]
                for n in range(self.rows) if n != i
            ]).determinant() * (-1)**(i+j)
                for j in range(self.cols)
            ]
            for i in range(self.rows)
        ])
        adj = minors.transpose()
        return adj.scalar_multiply(self.determinant()**-1)

In [3]:
M = Matrix([[1, 2, 5], [4, 5, 3], [7, 8, 9]])
M.inverse()

Matrix 3x3:
  / -0.8750  -0.9167   0.7917 \
  |  0.6250   1.0833  -0.7083 |
  \  0.1250  -0.2500   0.1250 /

In [4]:
M = np.array([[1, 2, 5], [4, 5, 3], [7, 8, 9]])
np.linalg.inv(M)

array([[-0.875     , -0.91666667,  0.79166667],
       [ 0.625     ,  1.08333333, -0.70833333],
       [ 0.125     , -0.25      ,  0.125     ]])